# 104 - Plantilla de Pipeline — **Soluciones**

Pipeline **ETL** con Prefect (si está disponible) y alternativa Python pura.

In [ ]:

from pathlib import Path
import pandas as pd

HAS_PREF=False
try:
    from prefect import flow, task
    HAS_PREF=True
    print("✅ Prefect disponible")
except Exception as e:
    print("ℹ Prefect no disponible:", e)

if HAS_PREF:
    @task
    def extract(path="data/dataset.csv"):
        try:
            return pd.read_csv(path)
        except Exception:
            # dataset sintético si falla
            return pd.DataFrame({"feature1":[0,1], "feature2":[1,-1], "target":[1,0]})
    @task
    def transform(df):
        return df.dropna()
    @task
    def load(df, path="artifacts/clean.csv"):
        Path("artifacts").mkdir(exist_ok=True)
        df.to_csv(path, index=False)
        return {"rows": len(df), "path": path}
    @flow
    def etl():
        d = extract()
        t = transform(d)
        return load(t)
    print("Ejecuta etl() para correr el pipeline con Prefect.")
else:
    def etl(path_in="data/dataset.csv", path_out="artifacts/clean.csv"):
        try:
            df = pd.read_csv(path_in)
        except Exception:
            df = pd.DataFrame({"feature1":[0,1], "feature2":[1,-1], "target":[1,0]})
        df = df.dropna()
        Path("artifacts").mkdir(exist_ok=True)
        df.to_csv(path_out, index=False)
        return {"rows": len(df), "path": path_out}
    print("Ejecuta etl() (versión Python pura).")
